In terminal do:

python -m venv venv (Create a Python virtual environment)

In PowerShell: venv\Scripts\Activate.ps1

python -m pip install -r requirements.txt

ollama pull llama3:8b

Source: https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide

To help with coding we used AI Text Generation LLM ChatGPT source: https://chatgpt.com

In [10]:
#import the needed modules
import ollama
import json
import random
import pandas as pd
from tqdm import tqdm

In [11]:
from datasets import load_dataset

# Load the dataset to train and test from https://huggingface.co/datasets/UniqueData/email-spam-classification
dataset = load_dataset("UniqueData/email-spam-classification", split="train")


#plit the data to a training and testing dataset
split_data = dataset.train_test_split(test_size=0.2, seed=1)

train_data = split_data["train"]
test_data = split_data["test"]

In [12]:
#in this function we choose 5 spam and 5 not spam emails from the training dataset
def build_examples(train_data, n_spam=5, n_ham=5, seed=1):
    spam = [row for row in train_data if row["type"] == "spam"] #split the spam and not spam emails into a list
    ham = [row for row in train_data if row["type"] == "not spam"]

    random.seed(seed) #by using a seed we get the same random values
    spam_samples = random.sample(spam, n_spam) #choose 5 spam emails randomly from the list
    ham_samples = random.sample(ham, n_ham)

    examples = []
    for row in spam_samples + ham_samples:
        score = 1 if row["type"] == "spam" else 0 #change the string value of spam to a number
        examples.append(  #prepare the emails for the prompt
            f"""Email:
\"\"\"{row['text']}\"\"\"
Spam score: {score}
"""
        )
    return "\n".join(examples) #put together the emails

In [13]:
Examples = build_examples(train_data) #call the function

#Following is the System Prompt for our Spam detector program
SYSTEM_PROMPT = f""" 
You are a spam detection system.

Spam score definition:
- Number between 0.0 and 1.0
- 0.0 = definitely not spam
- 1.0 = definitely spam
- Any number between 0.0 and 1.0 represents your confidence
- Only output 0 or 1 if you are fully certain
- Respond ONLY with valid JSON. The reasoning should only be one sentence, under 10 words.
{{
  "spam_score": float,
  "reasoning": string
}}

Below are labeled training examples:

Email:
"Win a free iPhone now!"
Spam score: 0.9

Email:
"Meeting agenda for tomorrow"
Spam score: 0.1

{Examples}

Now analyze the next email.

"""

In [14]:
#This function makes sure that we get a useful JSON value from the llama model

def check_json(raw_output):
    start = raw_output.find("{")
    end = raw_output.find("}")

    if start != -1 and end != -1 and end > start:
        return raw_output[start:end + 1]

    if end == -1:
        fixed = raw_output.strip() + "\n}"

    if start == -1:
        sc = fixed.find("spam_score")
        fixed = '{\n"' + fixed[sc:]
    
    return fixed

In [15]:
#we used the Ollama model "llama3:8b"
MODEL_NAME = "llama3:8b"

#The following function is where we classify the emails from the testing dataset

def classify_email(email_text, email_title):
    content = f"Title: {email_title}\n\nBody:\n{email_text}" # into a formatted string we put the title and text of the email
    
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT}, #tells the model the system prompt, system is to define rules, output
            {"role": "user", "content": f"Email:\n\"\"\"{content}\"\"\""} #the answer of the model, the triple quotes tell the LLM that inside is only the eamil text
        ]
    )
    
    raw = response["message"]["content"] #Here we get back the JSON file in a string format
    raw = check_json(raw)
    return json.loads(raw) #turning the string into a JSON

In [16]:
results = []

for row in tqdm(test_data):
    pred = classify_email(row["text"], email_title=row["title"]) #Here we put the email into the model

    score = pred["spam_score"] #getting the spam score from the JSON
    predicted_label = "spam" if score > 0.5 else "not spam" #decide based on the score whether spam or not
    recommendation = "discard" if score > 0.5 else "forward" #decide based on the score whether discard or forward

    #Put the results into a JSON
    results.append({
        "true_label": row["type"],
        "predicted_label": predicted_label,
        "spam_score": score,
        "reasoning": pred["reasoning"],
        "recommendation": recommendation, 
        "title": row["title"],
        "text": row["text"]
    })

#turn the JSON into a dataframe to make readabilty better
df = pd.DataFrame(results)
df["correct"] = df["true_label"] == df["predicted_label"] #decide if we got it right or not

accuracy = df["correct"].mean() #get the accuracy sum of correct divided by sum of all testing emails
print(f"Test accuracy: {accuracy:.2%}")

100%|██████████| 17/17 [07:37<00:00, 26.93s/it]

Test accuracy: 70.59%


In [17]:
#transform the results into a csv
df.to_csv("results.csv")